# GitHub Collaboration Analytics with Aura Analytics

This notebook uses **Neo4j Aura Analytics** with native graph projection - all processing runs in Neo4j.

## Business Hypotheses

| Metric | Business Meaning | Actionable Insight |
|--------|------------------|---------------------|
| **Degree** | How many collaborators someone works with | High = coordination hub |
| **PageRank** | Influence weighted by connections | High = key for information spread |
| **Betweenness** | How often you bridge groups | High = critical cross-team connector |
| **Community** | Natural working clusters | Team boundaries |

## Setup

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from graphdatascience import GdsSessions
from graphdatascience.session import AuraAPICredentials, DbmsConnectionInfo, SessionMemory
from datetime import timedelta
import pandas as pd

load_dotenv(override=True)

# Neo4j driver for queries
driver = GraphDatabase.driver(os.getenv('NEO4J_URI'), auth=(os.getenv('NEO4J_USERNAME'), os.getenv('NEO4J_PASSWORD')))

def query(cypher, **params):
    with driver.session(database=os.getenv('NEO4J_DATABASE', 'neo4j')) as session:
        return pd.DataFrame([dict(r) for r in session.run(cypher, params)])

# Aura Analytics session
gds = GdsSessions(api_credentials=AuraAPICredentials(
    client_id=os.getenv('AURA_CLIENT_ID'),
    client_secret=os.getenv('AURA_CLIENT_SECRET'),
    project_id=os.getenv('AURA_TENANT_ID')
)).get_or_create(
    "github-analytics",
    memory=SessionMemory.m_8GB,
    db_connection=DbmsConnectionInfo(
        uri=os.getenv('NEO4J_URI'),
        username=os.getenv('NEO4J_USERNAME'),
        password=os.getenv('NEO4J_PASSWORD'),
        database=os.getenv('NEO4J_DATABASE', 'neo4j')
    ),
    ttl=timedelta(hours=1)
)

print("✅ Connected to Neo4j and Aura Analytics")

✅ Connected to Neo4j and Aura Analytics


## Project the Collaboration Graph

Creates **User→User** edges (users who work on the same issues) entirely in Neo4j.  
No data pulled through Python - fully scalable.

In [2]:
# Drop existing graph if present
if gds.graph.exists("collab")["exists"]:
    gds.graph.drop(gds.graph.get("collab"))

# Project collaboration graph - runs entirely in Neo4j
G, stats = gds.graph.project(
    "collab",
    """
    MATCH (i:Issue)-[:ASSIGNED_TO]->(u1:User)
    MATCH (i)-[:ASSIGNED_TO]->(u2:User)
    WHERE u1 <> u2
    RETURN gds.graph.project.remote(u1, u2)
    """
)
print(f"✅ Projected {G.node_count():,} users, {G.relationship_count():,} collaboration edges")

 Graph creation from Triplets:  13%|#3        | 13.12/100 [00:00<?, ?%/s]

✅ Projected 112,468 users, 962,766 collaboration edges


## Run Algorithms & Write to Graph

In [3]:
gds.degree.write(G, writeProperty='degree')
print("✅ Wrote degree centrality")

gds.pageRank.write(G, writeProperty='pagerank')
print("✅ Wrote PageRank")

gds.louvain.write(G, writeProperty='community')
print("✅ Wrote community assignments")

Write-Back (graph: collab):   0%|          | 0.0/100 [00:00<?, ?%/s]

✅ Wrote degree centrality


Write-Back (graph: collab):   0%|          | 0.0/100 [00:00<?, ?%/s]

✅ Wrote PageRank


 Louvain:   7%|6         | 6.6/100 [00:00<?, ?%/s]

Write-Back (graph: collab):   0%|          | 0.0/100 [00:00<?, ?%/s]

✅ Wrote community assignments


## Query Results from Neo4j

In [4]:
print("🔗 TOP COLLABORATION HUBS (degree)\n")
query("""
    MATCH (u:User) WHERE u.degree IS NOT NULL
    RETURN u.username as user, u.degree as degree, round(u.pagerank, 4) as pagerank
    ORDER BY u.degree DESC LIMIT 10
""")

🔗 TOP COLLABORATION HUBS (degree)



,user,degree,pagerank
0,AutomationUIUser,8214.0,1.0117
1,AutomationSyncUser,7812.0,0.9670
2,Vishwa-gajjar,7246.0,0.9050
3,builderCE,6225.0,1.2408
4,ariCETester,5983.0,1.1907
5,scholokov,2246.0,5.4694
6,connieCE,1798.0,0.4522
7,SOLPLPARTY,1632.0,0.9612
8,amblerkr,1632.0,0.9612
9,ABatalov,1557.0,3.5159


In [5]:
print("⭐ MOST INFLUENTIAL (PageRank)\n")
query("""
    MATCH (u:User) WHERE u.pagerank IS NOT NULL
    RETURN u.username as user, round(u.pagerank, 4) as pagerank, u.degree as degree
    ORDER BY u.pagerank DESC LIMIT 10
""")

⭐ MOST INFLUENTIAL (PageRank)



,user,pagerank,degree
0,yanliang567,44.8981,778.0
1,dgilleland,23.0438,51.0
2,shufo,23.0438,60.0
3,Nikhil-Nandagopal,15.1270,935.0
4,FlashAnton,13.0091,256.0
5,matrix-meow,12.4877,338.0
6,camundait,12.2922,138.0
7,kelaja,12.1383,336.0
8,nayutah,12.0990,192.0
9,babyfengfjx,11.4149,513.0


In [6]:
print("👥 COLLABORATION COMMUNITIES\n")
query("""
    MATCH (u:User) WHERE u.community IS NOT NULL
    WITH u.community as community, collect(u.username)[0..5] as members, count(*) as size
    RETURN community, size, members
    ORDER BY size DESC LIMIT 10
""")

👥 COLLABORATION COMMUNITIES



,community,size,members
0,32908,611,"[cscheid, jamesrhester, dlebauer, blsqr, Chris..."
1,20767,502,"[kjduensing, nihil2501, erinrwhite, tejans24, ..."
2,56040,353,"[cead22, rinej, johnmlee101, badeggg, cleverjam]"
3,30384,301,"[dorchard, vgeorge, aaa34169, jemrobinson, con..."
4,52057,263,"[archerzz, rayluo, joaopgrassi, lmolkova, bill..."
5,11758,249,"[marc-portier, scanon, webyrd, chienchi, Shahi..."
6,68380,242,"[smortex, jkowall, Gaganjuneja, sumerjabri, je..."
7,58582,242,"[BigBlueHat, agropper, LeaVerou, goneall, nish..."
8,50985,237,"[alexzielenski, liggitt, smarterclayton, wilso..."
9,12716,208,"[kkaempf, manno, ly5156, thardeck, weyfonk]"


In [7]:
# Validate: show dominant repo for top 5 communities
# Proves communities = real organizational boundaries
print("🔍 COMMUNITY VALIDATION - What repos do members work on?\n")

# Get top 5 community IDs first
top_communities = query("""
    MATCH (u:User) WHERE u.community IS NOT NULL
    RETURN u.community as community, count(*) as size
    ORDER BY size DESC LIMIT 5
""")['community'].tolist()

# Then get their dominant repos (fast - only 5 communities)
query(f"""
    MATCH (u:User) WHERE u.community IN {top_communities}
    MATCH (i:Issue)-[:ASSIGNED_TO]->(u)
    MATCH (r:Repository)-[:HAS_ISSUE]->(i)
    WITH u.community as community, r.repository_full_name as repo, 
         count(DISTINCT u) as users
    ORDER BY community, users DESC
    WITH community, collect(repo)[0] as dominant_repo, collect(users)[0] as users_in_repo
    RETURN community, dominant_repo, users_in_repo
    ORDER BY users_in_repo DESC
""")

🔍 COMMUNITY VALIDATION - What repos do members work on?



,community,dominant_repo,users_in_repo
0,20767,department-of-veterans-affairs/va.gov-team,402
1,56040,Expensify/App,324
2,32908,openjournals/joss-reviews,299
3,52057,Azure/azure-sdk-for-net,58
4,30384,openjournals/joss-reviews,32


In [8]:
# Cleanup
gds.graph.drop(G)
driver.close()
print("✅ Done. Metrics persisted as node properties.")

✅ Done. Metrics persisted as node properties.
